## 1. Setup on a T4 runtime

In [ ]:
%pip install -q "ultralytics>=8.3,<9" "sahi>=0.11.20,<0.12" "gradio>=5,<6"
from pathlib import Path
import subprocess, sys, torch
from ultralytics import YOLO
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU'
repo = Path('/content/Small-Object-Detection')
if not repo.exists():
    subprocess.run(['git','clone','--branch','main','https://github.com/Ananyasaini06/Small-Object-Detection.git',str(repo)], check=True)
sys.path.insert(0, str(repo))

## 2. Mount Drive and recover a trusted VisDrone checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
project = drive_root / 'Small-Object-Detection'
project.mkdir(exist_ok=True)
# Set this to your own trusted VisDrone best.pt or last.pt if stored elsewhere.
manual = None
candidates = [Path(manual)] if manual else []
if manual and not candidates[0].is_file():
    raise FileNotFoundError(manual)
for folder in (project, drive_root/'P10_runs', Path('/content/P10_runs'), repo):
    for filename in ('best.pt', 'last.pt'):
        candidates.extend(sorted(folder.glob('**/final_baseline*/weights/'+filename), key=lambda p:p.stat().st_mtime, reverse=True))
        candidates.extend([folder/filename])
weights = next((p for p in candidates if p.is_file()), None)
if weights:
    detector = YOLO(str(weights))
    assert len(detector.names) == 10, 'Verify this is a trained VisDrone checkpoint'
print('Recovered:', weights)

## 3. Prepare VisDrone for training or evaluation

In [ ]:
from ultralytics.data.utils import check_det_dataset
data = check_det_dataset('VisDrone.yaml')
val_dir = Path(data['val'])
label_dir = val_dir.parent.parent / 'labels' / val_dir.name
assert val_dir.is_dir() and label_dir.is_dir()

## 4. Optional training if recovery found no weights

In [ ]:
TRAIN = False  # Set True only after checking Drive for your original weights.
if weights is None and TRAIN:
    model = YOLO('yolo11n.pt')
    model.train(data='VisDrone.yaml', epochs=15, imgsz=640, batch=8, device=0,
                workers=2, project=str(project), name='final_baseline', seed=42, patience=5)
    weights = Path(model.trainer.best)
print('Checkpoint:', weights)

## 5. Reproduce the corrected 100-image evaluation

In [ ]:
from evaluate import evaluate
assert weights and weights.is_file(), 'Recover or train a checkpoint first'
evaluate(str(weights), val_dir, label_dir, project/'evaluation')
# For the other historical tile sizes, repeat with tile=256 and tile=640.

## 6. Launch the upload demo

In [ ]:
from demo import launch
assert weights and weights.is_file(), 'Recover or train a checkpoint first'
launch(weights)